# Red Neuronal - Clasificacion Cancer de Mama

In [ ]:
#Importamos las librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Dataset utilizado: **Breast Cancer Wisconsin** de Kaggle

https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

El objetivo es predecir la columna **diagnosis**: M (Maligno) o B (Benigno).

In [ ]:
#Cargamos el dataset
df = pd.read_csv('data.csv')
df.head()

## Analisis exploratorio de datos

In [ ]:
#Analizamos el shape
df.shape

In [ ]:
#Eliminamos columnas que no sirven para el modelo
df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')
df.head()

In [ ]:
#Visualizamos los tipos de datos
df.dtypes

Resumen del dataset:
* El dataset tiene 569 filas y 31 columnas luego de la limpieza.
* Hay 30 variables numericas que describen caracteristicas del nucleo celular.
* La variable objetivo es **diagnosis**: M (Maligno) y B (Benigno).

In [ ]:
#Exploramos la variable objetivo
print(df['diagnosis'].value_counts())
print()
print(df['diagnosis'].value_counts(normalize=True).mul(100).round(2))

In [ ]:
#Grafico de la variable objetivo
df['diagnosis'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'], edgecolor='white')
plt.title('Distribucion de la variable objetivo')
plt.xlabel('Diagnosis')
plt.ylabel('Cantidad')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
#Verificamos valores nulos
df.isnull().sum()

## Procesamiento de datos

In [ ]:
#Separamos en X e y
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

print('Shape X:', X.shape)
print('Shape y:', y.shape)

In [ ]:
#Importamos las librerias necesarias
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

#Codificamos la variable objetivo: B=0, M=1
le = LabelEncoder()
y_enc = le.fit_transform(y)

print('Clases:', le.classes_)

In [ ]:
#30% para test y 70% para train
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.30, random_state=42, stratify=y_enc
)

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)

In [ ]:
#Veamos que obtuvimos en el split
print('Distribucion y_train:')
print(pd.Series(y_train).value_counts())

In [ ]:
#Escalamos los datos - necesario para que la red neuronal funcione bien
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [ ]:
#Veamos que tenemos en X_train luego del escalado
pd.DataFrame(X_train_sc, columns=X.columns).head()

## Entrenamiento del modelo - Red Neuronal

In [ ]:
#Importamos tensorflow y keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

#Fijamos semilla para reproducibilidad
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
#Creacion de la red neuronal
model = keras.Sequential([
    layers.Input(shape=(X_train_sc.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1,  activation='sigmoid')
])

model.summary()

In [ ]:
#Compilamos el modelo
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#Entrenamiento
history = model.fit(
    X_train_sc, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    verbose=0
)

print('Entrenamiento completado')

In [ ]:
#Calculo de predicciones en Train y Test
y_prob_train = model.predict(X_train_sc, verbose=0).flatten()
y_prob_test  = model.predict(X_test_sc,  verbose=0).flatten()

#Convertimos probabilidad a clase con umbral 0.5
y_pred_train = (y_prob_train >= 0.5).astype(int)
y_pred_test  = (y_prob_test  >= 0.5).astype(int)

## Evaluacion del modelo

In [ ]:
#Importamos las metricas
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_curve, auc,
                             precision_score, recall_score, f1_score)

#Calculo el accuracy en Train
acc_train = accuracy_score(y_train, y_pred_train)
print('Accuracy Train:', round(acc_train * 100, 2), '%')

#Calculo el accuracy en Test
acc_test = accuracy_score(y_test, y_pred_test)
print('Accuracy Test: ', round(acc_test  * 100, 2), '%')

In [ ]:
#Reporte de clasificacion completo
print(classification_report(y_test, y_pred_test, target_names=['Benigno', 'Maligno']))

In [ ]:
#Curvas de aprendizaje del modelo
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'],     label='Train')
plt.plot(history.history['val_loss'], label='Validacion', linestyle='--')
plt.title('Curva de perdida')
plt.xlabel('Epocas')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'],     label='Train')
plt.plot(history.history['val_accuracy'], label='Validacion', linestyle='--')
plt.title('Curva de accuracy')
plt.xlabel('Epocas')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
#Matriz de confusion
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benigno', 'Maligno'],
            yticklabels=['Benigno', 'Maligno'],
            linewidths=2, linecolor='white')
plt.title('Matriz de Confusion')
plt.ylabel('Real')
plt.xlabel('Prediccion')
plt.tight_layout()
plt.show()

In [ ]:
#Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='purple', lw=2, label='AUC = ' + str(round(roc_auc, 4)))
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#Resumen final de metricas
print('Accuracy : ', round(acc_test, 4))
print('Precision:', round(precision_score(y_test, y_pred_test), 4))
print('Recall   :', round(recall_score(y_test, y_pred_test), 4))
print('F1-Score :', round(f1_score(y_test, y_pred_test), 4))
print('ROC-AUC  :', round(roc_auc, 4))